# Qiskit Random Circuit Sweep vs LSQECC Runtime

This notebook generates random circuits as a function of `num_qubits`, runs the LSQECC slicer executable, and plots runtime and selected compiler statistics.

In [ ]:
!uv pip install -q qiskit pandas matplotlib

In [ ]:
import re
import time
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.circuit.library import HGate, TGate, CXGate
from qiskit.circuit.exceptions import CircuitError
from qiskit.qasm2 import dumps

In [ ]:
def qiskit_random_circuit(num_qubits, depth, max_operands=2, seed=None):
    if max_operands < 1 or max_operands > 2:
        raise CircuitError("Parameter max_operands must be between 1 and 2")

    one_q_ops = [HGate, TGate]
    two_q_ops = [CXGate]

    qc = QuantumCircuit(num_qubits)

    if seed is None:
        seed = np.random.randint(0, np.iinfo(np.int32).max)

    rng = np.random.default_rng(seed)

    for _ in range(depth):
        remaining_qubits = list(range(num_qubits))

        while remaining_qubits:
            max_possible_operands = min(len(remaining_qubits), max_operands)
            num_operands = rng.integers(1, max_possible_operands + 1)

            rng.shuffle(remaining_qubits)
            operands = remaining_qubits[:num_operands]
            remaining_qubits = [q for q in remaining_qubits if q not in operands]

            if num_operands == 1:
                gate_class = rng.choice(one_q_ops)
                gate = gate_class()
                qc.append(gate, [operands[0]])
            elif num_operands == 2:
                gate_class = rng.choice(two_q_ops)
                gate = gate_class()
                qc.append(gate, operands)

    return qc


def save_to_qasm20(circuit, output_path):
    output_path = Path(output_path)
    output_path.write_text(dumps(circuit))

In [ ]:
ROOT = Path.cwd()
SLICER_CANDIDATES = [
    ROOT / "build" / "lsqecc_slicer_main",
    ROOT / "build" / "lsqecc_slicer",
    ROOT / "build" / "bin" / "lsqecc_slicer_main",
    ROOT / "build" / "bin" / "lsqecc_slicer",
    ROOT / "lsqecc_slicer",
]

SLICER_BIN = next((p for p in SLICER_CANDIDATES if p.exists()), None)
if SLICER_BIN is None:
    raise FileNotFoundError(
        "Could not find lsqecc slicer executable. Build project first (e.g., cmake --build build)."
    )

ARTIFACT_DIR = ROOT / "benchmark_artifacts"
QASM_DIR = ARTIFACT_DIR / "qasm"
ARTIFACT_DIR.mkdir(exist_ok=True)
QASM_DIR.mkdir(exist_ok=True)

print(f"Using slicer executable: {SLICER_BIN}")
print(f"QASM output directory: {QASM_DIR}")

In [ ]:
STATS_PATTERNS = {
    "ls_instructions_read": re.compile(r"LS\s+Instructions\s+read\s+(\d+)"),
    "slices_count": re.compile(r"Slices\s+(\d+)"),
    "total_volume": re.compile(r"Total volume:\s*(\d+)"),
    "distillation_volume": re.compile(r"Distillation volume:\s*(\d+)"),
    "unused_routing_volume": re.compile(r"Unused routing volume:\s*(\d+)"),
    "dead_volume": re.compile(r"Dead volume:\s*(\d+)"),
    "other_active_volume": re.compile(r"Other active volume:\s*(\d+)"),
    "patch_compute_time_s": re.compile(r"Made patch computation\. Took\s*([0-9.eE+\-]+)s"),
}


def parse_slicer_stats(stdout_text):
    parsed = {}
    for key, pattern in STATS_PATTERNS.items():
        m = pattern.search(stdout_text)
        if m:
            value = m.group(1)
            parsed[key] = float(value) if "time" in key else int(value)
        else:
            parsed[key] = np.nan
    return parsed


def run_slicer_on_qasm(qasm_path, layout_file, timeout_s=None):
    cmd = [
        str(SLICER_BIN),
        "-q",
        "-i",
        str(qasm_path),
        "--noslices",
        "-l",
        layout_file,
        "-f",
        "stats",
        "--graceful",
    ]

    t0 = time.perf_counter()
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_s)
        elapsed_s = time.perf_counter() - t0
        stdout_text = result.stdout
        stderr_text = result.stderr
        returncode = result.returncode
        timed_out = False
    except subprocess.TimeoutExpired as exc:
        elapsed_s = time.perf_counter() - t0
        stdout_text = exc.stdout or ""
        stderr_text = (exc.stderr or "") + f"\nTimed out after {timeout_s} seconds"
        returncode = -1
        timed_out = True

    parsed = parse_slicer_stats(stdout_text)
    parsed.update({
        "returncode": returncode,
        "wall_time_s": elapsed_s,
        "timed_out": timed_out,
        "stdout": stdout_text,
        "stderr": stderr_text,
    })
    return parsed

In [ ]:
# Sweep setup: adjust as desired.
qubit_start = 5
qubit_step = 5
cycles_per_n = 1
max_benchmark_timeout_min = 100
max_benchmark_timeout_s = max_benchmark_timeout_min * 60  # Total benchmark budget in seconds
max_cycle_timeout_s = max_benchmark_timeout_s             # Timeout for each slicer invocation
max_operands = 2
seed_base = 42
layout_file = "./layout_80x80.txt"
depth_for = lambda n: n ** 3

records = []
benchmark_start = time.perf_counter()
n = qubit_start

print(
    f"Starting indefinite qubit sweep from n={qubit_start} in steps of {qubit_step}. "
    f"Stops after {max_benchmark_timeout_s // 60} minutes (or manual interrupt)."
)

try:
    while True:
        elapsed_total_s = time.perf_counter() - benchmark_start
        if elapsed_total_s >= max_benchmark_timeout_s:
            print(f"Reached total benchmark timeout after {elapsed_total_s:.1f}s. Stopping sweep.")
            break

        depth = int(depth_for(n))
        print(
            f"Running {cycles_per_n} cycles on {n} qubits "
            f"(elapsed {elapsed_total_s:.1f}s/{max_benchmark_timeout_s}s)"
        )

        for cycle in range(cycles_per_n):
            elapsed_total_s = time.perf_counter() - benchmark_start
            if elapsed_total_s >= max_benchmark_timeout_s:
                print(
                    f"Reached total benchmark timeout during n={n}, cycle={cycle + 1}. "
                    "Stopping sweep."
                )
                break

            seed = seed_base + n * 1000 + cycle

            circuit = qiskit_random_circuit(
                num_qubits=n,
                depth=depth,
                max_operands=max_operands,
                seed=seed,
            )

            qasm_path = QASM_DIR / f"random_q{n}_c{cycle}_d{depth}.qasm"
            save_to_qasm20(circuit, qasm_path)

            stats = run_slicer_on_qasm(qasm_path, layout_file=layout_file, timeout_s=max_cycle_timeout_s)
            stats.update({
                "num_qubits": n,
                "cycle": cycle,
                "depth": depth,
                "seed": seed,
                "qasm_path": str(qasm_path),
                "timeout_s": max_cycle_timeout_s,
                "benchmark_elapsed_s": elapsed_total_s,
            })
            records.append(stats)

            status = "timed out" if stats["timed_out"] else f"rc={stats['returncode']}"
            print(
                f"n={n:>3}, cycle={cycle + 1:>2}/{cycles_per_n}, depth={depth:>6}, "
                f"{status}, wall={stats['wall_time_s']:.4f}s, total_volume={stats['total_volume']}, "
                f"ls_instr={stats['ls_instructions_read']}, slices={stats['slices_count']}"
            )
        else:
            n += qubit_step
            continue

        break

except KeyboardInterrupt:
    print("Manual interrupt received. Finalizing results collected so far.")

result_columns = [
    "num_qubits",
    "cycle",
    "depth",
    "seed",
    "qasm_path",
    "timeout_s",
    "benchmark_elapsed_s",
    "timed_out",
    "returncode",
    "wall_time_s",
    "patch_compute_time_s",
    "ls_instructions_read",
    "slices_count",
    "total_volume",
    "distillation_volume",
    "unused_routing_volume",
    "dead_volume",
    "other_active_volume",
    "stdout",
    "stderr",
]

df = pd.DataFrame(records)
if df.empty:
    df = pd.DataFrame(columns=result_columns)
else:
    df = df.sort_values(["num_qubits", "cycle"]).reset_index(drop=True)

display(df[[
    "num_qubits",
    "cycle",
    "depth",
    "timeout_s",
    "benchmark_elapsed_s",
    "timed_out",
    "returncode",
    "wall_time_s",
    "patch_compute_time_s",
    "ls_instructions_read",
    "slices_count",
    "total_volume",
    "distillation_volume",
    "unused_routing_volume",
    "dead_volume",
    "other_active_volume",
]])

In [ ]:
failed = df[df["returncode"] != 0]
if not failed.empty:
    print("Some runs failed. First few stderr snippets:")
    for _, row in failed.head(3).iterrows():
        print(f"\nnum_qubits={row['num_qubits']} stderr:")
        print((row['stderr'] or "")[:600])
else:
    print("All runs succeeded.")

In [ ]:
plot_df = df[df["returncode"] == 0].copy()
if plot_df.empty:
    print("No successful runs to plot yet.")
else:
    summary_df = (
        plot_df
        .groupby("num_qubits")
        .agg(
            wall_time_mean=("wall_time_s", "mean"),
            wall_time_min=("wall_time_s", "min"),
            wall_time_max=("wall_time_s", "max"),
            patch_time_mean=("patch_compute_time_s", "mean"),
            patch_time_min=("patch_compute_time_s", "min"),
            patch_time_max=("patch_compute_time_s", "max"),
            total_volume_mean=("total_volume", "mean"),
            total_volume_min=("total_volume", "min"),
            total_volume_max=("total_volume", "max"),
            distillation_volume_mean=("distillation_volume", "mean"),
            distillation_volume_min=("distillation_volume", "min"),
            distillation_volume_max=("distillation_volume", "max"),
            unused_routing_volume_mean=("unused_routing_volume", "mean"),
            unused_routing_volume_min=("unused_routing_volume", "min"),
            unused_routing_volume_max=("unused_routing_volume", "max"),
            dead_volume_mean=("dead_volume", "mean"),
            dead_volume_min=("dead_volume", "min"),
            dead_volume_max=("dead_volume", "max"),
            other_active_volume_mean=("other_active_volume", "mean"),
            other_active_volume_min=("other_active_volume", "min"),
            other_active_volume_max=("other_active_volume", "max"),
            ls_instructions_read_mean=("ls_instructions_read", "mean"),
            ls_instructions_read_min=("ls_instructions_read", "min"),
            ls_instructions_read_max=("ls_instructions_read", "max"),
            slices_count_mean=("slices_count", "mean"),
            slices_count_min=("slices_count", "min"),
            slices_count_max=("slices_count", "max"),
        )
        .reset_index()
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    wall_time_yerr = np.vstack([
        summary_df["wall_time_mean"] - summary_df["wall_time_min"],
        summary_df["wall_time_max"] - summary_df["wall_time_mean"],
    ])
    line_runtime = axes[0].errorbar(
        summary_df["num_qubits"],
        summary_df["wall_time_mean"],
        yerr=wall_time_yerr,
        marker="o",
        capsize=4,
        label="Wall time mean +/- range",
    )

    qubit_step_vals = summary_df["num_qubits"].diff().dropna()
    bar_width = 0.65 * (float(qubit_step_vals.min()) if not qubit_step_vals.empty else 1.0)
    ax0_right = axes[0].twinx()
    bars_total = ax0_right.bar(
        summary_df["num_qubits"],
        summary_df["total_volume_mean"],
        width=bar_width,
        alpha=0.28,
        color="tab:orange",
        label="Total volume mean",
    )

    axes[0].set_title("Runtime and Total Volume vs Number of Qubits")
    axes[0].set_xlabel("num_qubits")
    axes[0].set_ylabel("seconds")
    ax0_right.set_ylabel("total volume")
    axes[0].grid(True, alpha=0.3)

    handles = [line_runtime.lines[0], bars_total]
    labels = ["Wall time mean +/- range", "Total volume mean"]
    axes[0].legend(handles, labels, loc="upper left")

    volume_series = [
        ("distillation_volume", "Distillation volume"),
        ("unused_routing_volume", "Unused routing volume"),
        ("other_active_volume", "Other active volume"),
        ("ls_instructions_read", "LS Instructions read"),
        ("slices_count", "Slices count"),
    ]
    for col_name, label in volume_series:
        yerr = np.vstack([
            summary_df[f"{col_name}_mean"] - summary_df[f"{col_name}_min"],
            summary_df[f"{col_name}_max"] - summary_df[f"{col_name}_mean"],
        ])
        axes[1].errorbar(
            summary_df["num_qubits"],
            summary_df[f"{col_name}_mean"],
            yerr=yerr,
            marker="o",
            capsize=4,
            label=label,
        )
    axes[1].set_title("Circuit Statistics vs Number of Qubits")
    axes[1].set_xlabel("num_qubits")
    axes[1].set_ylabel("value")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(loc="upper left")

    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / "plot.png")
    plt.show()

In [ ]:
csv_path = ARTIFACT_DIR / "random_circuit_slicer_benchmark.csv"
df.to_csv(csv_path, index=False)
print(f"Saved results to: {csv_path}")